# PodForge — VoxCPM2 TTS Server (Kaggle)

This notebook runs VoxCPM2 on Kaggle's free T4 GPU.

**Steps:**
1. Turn on GPU: Settings → Accelerator → T4 GPU
2. Run all cells
3. Copy the ngrok public URL
4. Set it as `TTS_BASE_URL` in PodForge backend

**Kaggle free tier**: 30h/week GPU, resets every Saturday 00:00 UTC

In [ ]:
# Install dependencies
!pip install -q voxcpm fastapi uvicorn pyngrok soundfile numpy

In [ ]:
# Configure ngrok (get free token at https://dashboard.ngrok.com/get-started/your-authtoken)
from pyngrok import ngrok
ngrok.set_auth_token("3EOr384hN9MABHuFYtoCMY6Ujb5_6UDb37kBKe1Mx6E5DHUr8")
print("ngrok configured ✓")

In [ ]:
# Verify GPU is available
!nvidia-smi

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No GPU! Go to Settings → Accelerator → T4 GPU")

In [ ]:
# Download and load VoxCPM2 model (~4GB, first run only)
from voxcpm import VoxCPM

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

In [ ]:
# Start TTS server with ngrok tunnel
import io
import base64
import threading
import time
import subprocess
import numpy as np
import soundfile as sf
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn

# Auto-load model if not already loaded
try:
    model
    print("Model already loaded ✓")
except NameError:
    print("Loading model...")
    from voxcpm import VoxCPM
    model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
    print(f"Model loaded! Sample rate: {model.tts_model.sample_rate}")

app = FastAPI(title="PodForge TTS Server")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class TTSRequest(BaseModel):
    text: str
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    reference_wav_path: str | None = None

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/generate")
async def generate(req: TTSRequest):
    kwargs = {
        "text": req.text,
        "cfg_value": req.cfg_value,
        "inference_timesteps": req.inference_timesteps,
    }
    if req.reference_wav_path:
        kwargs["reference_wav_path"] = req.reference_wav_path

    wav = model.generate(**kwargs)
    buf = io.BytesIO()
    sf.write(buf, wav, model.tts_model.sample_rate, format="WAV")
    buf.seek(0)
    audio_b64 = base64.b64encode(buf.read()).decode()

    return {
        "audio_base64": audio_b64,
        "sample_rate": model.tts_model.sample_rate,
        "duration_seconds": len(wav) / model.tts_model.sample_rate,
    }

# Kill any existing process on port 8809
subprocess.run(["fuser", "-k", "8809/tcp"], capture_output=True)
time.sleep(1)

# Disconnect any existing ngrok tunnels
ngrok.kill()

# Start ngrok tunnel
public_url = ngrok.connect(8809)
print(f"\n{'='*60}")
print(f"PUBLIC URL: {public_url}")
print(f"Copy this URL and set it as TTS_BASE_URL in PodForge!")
print(f"{'='*60}\n")

# Run server
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8809)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

print("TTS server is running! Keep this cell running.")
print("Press stop button to shut down.")

# Keep alive
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.kill()